# Vincent van Gogh — Color Analysis Pipeline

This notebook analyses a collection of Vincent van Gogh paintings using color-based computer vision techniques.

Main goals:
- load image metadata from `vgdb_2016.csv`,
- filter Vincent van Gogh paintings,
- download/cache images from `ImageURL`,
- extract dominant color palettes and color characteristics,
- find color-similar paintings,
- cluster paintings based on color features,
- visualise the results for the project presentation and final report.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import time
import hashlib
import requests
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

CSV_PATH = Path("vgdb_2016.csv")
IMAGE_DIR = Path("images_vangogh")
OUTPUT_DIR = Path("output_vangogh")

IMAGE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
MAX_IMAGES = 350  # set to None to use all Van Gogh images
N_PALETTE_COLORS = 6
N_CLUSTERS = 6

print("CSV exists:", CSV_PATH.exists(), CSV_PATH.resolve())

## 2. Load metadata and filter Vincent van Gogh

In [ ]:
df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
df["artist_normalized"] = df["Artist"].astype(str).str.lower().str.strip()

vangogh_df = df[df["artist_normalized"] == "vincent van gogh"].copy()
vangogh_df = vangogh_df.dropna(subset=["ImageURL"]).reset_index(drop=True)

print("Van Gogh records with ImageURL:", len(vangogh_df))
display(vangogh_df[["Artist", "ImageURL", "PixelWidth", "PixelHeight", "PaintingID"]].head())

In [ ]:
# If this returns unexpected values, inspect alternative artist spellings
df[df["Artist"].astype(str).str.lower().str.contains("gogh", na=False)]["Artist"].value_counts()

## 3. Download/cache images

The notebook uses local caching: each image is downloaded once into `images_vangogh/`.

You *could* load images directly from URL every time, but caching is better because:
- repeated runs are much faster,
- network errors do not break the analysis,
- Wikimedia may throttle many repeated requests,
- clustering and palette extraction repeatedly open many images.

In [ ]:
def safe_filename_from_url(url: str, index: int) -> str:
    suffix = Path(str(url).split("?")[0]).suffix.lower()
    if suffix not in [".jpg", ".jpeg", ".png", ".webp"]:
        suffix = ".jpg"
    # Include hash to avoid collisions
    url_hash = hashlib.sha1(str(url).encode("utf-8")).hexdigest()[:10]
    return f"vangogh_{index:04d}_{url_hash}{suffix}"

def download_image(url: str, output_path: Path, timeout: int = 30) -> bool:
    try:
        response = requests.get(
            url,
            timeout=timeout,
            headers={"User-Agent": "Mozilla/5.0 (educational image analysis project)"}
        )
        content_type = response.headers.get("Content-Type", "")
        if response.status_code == 200 and "image" in content_type:
            output_path.write_bytes(response.content)
            return True
        return False
    except Exception:
        return False

work_df = vangogh_df.copy()
if MAX_IMAGES is not None:
    work_df = work_df.head(MAX_IMAGES).copy()

work_df = work_df.reset_index(drop=True)

local_paths = []
failed = []

for i, row in tqdm(work_df.iterrows(), total=len(work_df)):
    url = row["ImageURL"]
    output_path = IMAGE_DIR / safe_filename_from_url(url, i)

    if output_path.exists() and output_path.stat().st_size > 0:
        local_paths.append(output_path)
        continue

    ok = download_image(url, output_path)
    if ok:
        local_paths.append(output_path)
    else:
        local_paths.append(None)
        failed.append(url)

    time.sleep(0.05)

work_df["image_path"] = local_paths
work_df["image_exists"] = work_df["image_path"].notna()

vangogh_images_df = work_df[work_df["image_exists"]].copy().reset_index(drop=True)

print("Images available:", len(vangogh_images_df))
print("Failed downloads:", len(failed))
display(vangogh_images_df[["Artist", "ImageURL", "image_path", "PixelWidth", "PixelHeight", "PaintingID"]].head())

## 4. Validate and preview images

In [ ]:
valid_rows = []
invalid_paths = []

for _, row in tqdm(vangogh_images_df.iterrows(), total=len(vangogh_images_df)):
    try:
        with Image.open(row["image_path"]) as img:
            img.verify()
        valid_rows.append(row)
    except Exception:
        invalid_paths.append(row["image_path"])

vangogh_images_df = pd.DataFrame(valid_rows).reset_index(drop=True)

print("Valid images:", len(vangogh_images_df))
print("Invalid images:", len(invalid_paths))

In [ ]:
sample_df = vangogh_images_df.sample(min(12, len(vangogh_images_df)), random_state=RANDOM_STATE)

plt.figure(figsize=(14, 8))
for i, (_, row) in enumerate(sample_df.iterrows()):
    img = Image.open(row["image_path"]).convert("RGB")
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Image {i+1}", fontsize=8)

plt.tight_layout()
plt.show()

## 5. Color feature extraction

For each painting, this section extracts:
- dominant color palette,
- average RGB values,
- average hue, saturation, brightness,
- brightness contrast,
- image size/aspect ratio metadata.

In [ ]:
def load_image_for_analysis(path, max_size=256):
    img = Image.open(path).convert("RGB")
    img = ImageOps.exif_transpose(img)
    img.thumbnail((max_size, max_size))
    return img

def dominant_palette(img, n_colors=6, random_state=42):
    arr = np.asarray(img).reshape(-1, 3)
    # sample pixels for speed
    if len(arr) > 20000:
        idx = np.random.default_rng(random_state).choice(len(arr), size=20000, replace=False)
        arr_sample = arr[idx]
    else:
        arr_sample = arr

    kmeans = KMeans(n_clusters=n_colors, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(arr_sample)
    centers = kmeans.cluster_centers_.astype(int)

    counts = np.bincount(labels, minlength=n_colors)
    order = np.argsort(counts)[::-1]

    centers = centers[order]
    proportions = counts[order] / counts.sum()
    return centers, proportions

def color_characteristics(img):
    arr = np.asarray(img).astype(np.float32) / 255.0

    mean_rgb = arr.reshape(-1, 3).mean(axis=0)

    hsv = np.asarray(img.convert("HSV")).astype(np.float32)
    hue = hsv[:, :, 0] / 255.0
    saturation = hsv[:, :, 1] / 255.0
    brightness = hsv[:, :, 2] / 255.0

    return {
        "mean_r": mean_rgb[0],
        "mean_g": mean_rgb[1],
        "mean_b": mean_rgb[2],
        "mean_hue": hue.mean(),
        "mean_saturation": saturation.mean(),
        "mean_brightness": brightness.mean(),
        "brightness_std": brightness.std(),
        "contrast": brightness.std(),
    }

def palette_to_feature_vector(palette, proportions):
    # Weighted dominant colors as a compact representation
    weighted = (palette / 255.0) * proportions[:, None]
    return weighted.flatten()

def plot_palette(colors, proportions=None, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 1))
    colors_norm = np.array(colors) / 255.0
    if proportions is None:
        proportions = np.ones(len(colors)) / len(colors)
    start = 0
    for color, prop in zip(colors_norm, proportions):
        ax.barh([0], [prop], left=start, color=color, height=1)
        start += prop
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([])
    if title:
        ax.set_title(title, fontsize=9)
    return ax

In [ ]:
records = []
palette_vectors = []
palettes = []
palette_props = []

for idx, row in tqdm(vangogh_images_df.iterrows(), total=len(vangogh_images_df)):
    img = load_image_for_analysis(row["image_path"])
    palette, proportions = dominant_palette(img, n_colors=N_PALETTE_COLORS, random_state=RANDOM_STATE)
    characteristics = color_characteristics(img)

    record = row.to_dict()
    record.update(characteristics)
    record["aspect_ratio"] = row["PixelWidth"] / row["PixelHeight"] if row["PixelHeight"] else np.nan

    records.append(record)
    palettes.append(palette)
    palette_props.append(proportions)
    palette_vectors.append(palette_to_feature_vector(palette, proportions))

features_df = pd.DataFrame(records)
palette_vectors = np.vstack(palette_vectors)

print("Feature table shape:", features_df.shape)
display(features_df[["mean_r", "mean_g", "mean_b", "mean_hue", "mean_saturation", "mean_brightness", "contrast"]].head())

## 6. Visualise dominant palettes

In [ ]:
# Global average palette across sampled paintings
all_palette_colors = np.vstack(palettes)
all_palette_props = np.hstack(palette_props)

# Cluster all extracted dominant colors into a global palette
kmeans_global = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init=10)
labels_global = kmeans_global.fit_predict(all_palette_colors, sample_weight=all_palette_props)
global_colors = kmeans_global.cluster_centers_.astype(int)

counts = np.bincount(labels_global, weights=all_palette_props, minlength=8)
order = np.argsort(counts)[::-1]
global_colors = global_colors[order]
global_props = counts[order] / counts.sum()

fig, ax = plt.subplots(figsize=(9, 1.5))
plot_palette(global_colors, global_props, title="Global dominant color palette", ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "global_average_palette.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Palette grid for random sample
sample_indices = np.random.default_rng(RANDOM_STATE).choice(len(features_df), size=min(12, len(features_df)), replace=False)

fig, axes = plt.subplots(len(sample_indices), 2, figsize=(9, len(sample_indices) * 1.2))

if len(sample_indices) == 1:
    axes = np.array([axes])

for ax_row, idx in zip(axes, sample_indices):
    img = Image.open(features_df.loc[idx, "image_path"]).convert("RGB")
    ax_row[0].imshow(img)
    ax_row[0].axis("off")
    ax_row[1] = plot_palette(palettes[idx], palette_props[idx], title=f"Painting {idx}", ax=ax_row[1])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dominant_palette_grid.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Color similarity search

In [ ]:
# Similarity based on color palette feature vectors
scaler = StandardScaler()
X_color = scaler.fit_transform(palette_vectors)

sim_matrix = cosine_similarity(X_color)
np.fill_diagonal(sim_matrix, -1)

pairs = []
for i in range(len(sim_matrix)):
    j = sim_matrix[i].argmax()
    pairs.append({
        "image_a_index": i,
        "image_b_index": j,
        "similarity": sim_matrix[i, j],
        "image_a_path": features_df.loc[i, "image_path"],
        "image_b_path": features_df.loc[j, "image_path"],
        "image_a_url": features_df.loc[i, "ImageURL"],
        "image_b_url": features_df.loc[j, "ImageURL"],
    })

pairs_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False).drop_duplicates(subset=["image_b_index"]).head(20)

display(pairs_df.head(10))
pairs_df.to_csv(OUTPUT_DIR / "most_similar_color_pairs.csv", index=False)

In [ ]:
# Visualise top similar pairs
top_pairs = pairs_df.head(8)
fig, axes = plt.subplots(len(top_pairs), 2, figsize=(8, len(top_pairs) * 3))

if len(top_pairs) == 1:
    axes = np.array([axes])

for row_axes, (_, pair) in zip(axes, top_pairs.iterrows()):
    img_a = Image.open(pair["image_a_path"]).convert("RGB")
    img_b = Image.open(pair["image_b_path"]).convert("RGB")

    row_axes[0].imshow(img_a)
    row_axes[0].axis("off")
    row_axes[0].set_title(f"A: {pair['image_a_index']}")

    row_axes[1].imshow(img_b)
    row_axes[1].axis("off")
    row_axes[1].set_title(f"B: {pair['image_b_index']} | sim={pair['similarity']:.2f}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "most_similar_pairs_contact_sheet.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Clustering and 2D visualisation

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
features_df["cluster"] = kmeans.fit_predict(X_color)

cluster_summary = features_df.groupby("cluster")[["mean_saturation", "mean_brightness", "contrast"]].mean().round(3)
cluster_summary["count"] = features_df["cluster"].value_counts().sort_index()

display(cluster_summary)
cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary.csv")

In [ ]:
# PCA map
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_color)
features_df["pca_1"] = X_pca[:, 0]
features_df["pca_2"] = X_pca[:, 1]

plt.figure(figsize=(9, 7))
scatter = plt.scatter(features_df["pca_1"], features_df["pca_2"], c=features_df["cluster"], s=35)
plt.title("PCA map of Van Gogh paintings based on color features")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_similarity_map.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# t-SNE map
perplexity = min(30, max(5, len(X_color) // 10))
tsne = TSNE(n_components=2, perplexity=perplexity, learning_rate="auto", init="pca", random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X_color)
features_df["tsne_1"] = X_tsne[:, 0]
features_df["tsne_2"] = X_tsne[:, 1]

plt.figure(figsize=(9, 7))
scatter = plt.scatter(features_df["tsne_1"], features_df["tsne_2"], c=features_df["cluster"], s=35)
plt.title("t-SNE map of Van Gogh paintings based on color features")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_similarity_map.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Cluster contact sheets

In [ ]:
def make_contact_sheet(paths, thumb_size=(160, 160), columns=5):
    thumbs = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        img.thumbnail(thumb_size)
        canvas = Image.new("RGB", thumb_size, "white")
        x = (thumb_size[0] - img.width) // 2
        y = (thumb_size[1] - img.height) // 2
        canvas.paste(img, (x, y))
        thumbs.append(canvas)

    rows = int(np.ceil(len(thumbs) / columns))
    sheet = Image.new("RGB", (columns * thumb_size[0], rows * thumb_size[1]), "white")

    for idx, thumb in enumerate(thumbs):
        x = (idx % columns) * thumb_size[0]
        y = (idx // columns) * thumb_size[1]
        sheet.paste(thumb, (x, y))

    return sheet

cluster_dir = OUTPUT_DIR / "clusters"
cluster_dir.mkdir(exist_ok=True)

for cluster_id in sorted(features_df["cluster"].unique()):
    cluster_paths = features_df[features_df["cluster"] == cluster_id]["image_path"].head(20).tolist()
    sheet = make_contact_sheet(cluster_paths, columns=5)
    out_path = cluster_dir / f"cluster_{cluster_id}.png"
    sheet.save(out_path)
    display(sheet)
    print("Saved:", out_path)

## 10. Export final results

In [ ]:
features_out = OUTPUT_DIR / "color_features_with_clusters.csv"
features_df.to_csv(features_out, index=False)

print("Saved feature table:", features_out)
print("Saved outputs in:", OUTPUT_DIR.resolve())
print("Files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(OUTPUT_DIR))

## 11. Optional interpretation notes

Use the generated plots to answer these questions in the presentation/report:

- Which paintings are most similar in terms of color palette?
- Do the clusters have visible color differences?
- Are some clusters warmer, colder, brighter, darker, or more saturated?
- Does the 2D map show clear groups or gradual transitions?

This notebook currently uses color-based features only. A later extension can add CLIP or CNN image embeddings for semantic/style similarity beyond color.